# Convert GTFS `.txt` files to `.xlsx`

This notebook reads GTFS `.txt` files from a configurable data folder.

Priority:
1. `GTFS_DATA_DIR` environment variable (if defined).
2. Relative default from the repository root: `.src/gtfs/data` (when running notebook from repo root).

It exports tables to Excel in a clean and reproducible way.

It includes:
1. One `.xlsx` file per `.txt` table.
2. One optional combined `.xlsx` file with one sheet per table.

## Notes

- Set `TXT_FILE_NAME` in the configuration cell to convert a single `.txt` file.
- Set `TXT_FILE_NAME = None` to convert all `.txt` files in the data folder.
- Large files are split automatically into multiple `.xlsx` files, each staying within Excel's row limit.
- If you also run the combined workbook cell, oversized tables are split into multiple sheets automatically.
- Output location:
  - `...\.src\gtfs\data\excel_exports`

In [1]:
from pathlib import Path
import math
import os
import pandas as pd

# Find the project root and data folder.
# The data is always at: <project_root>/.src/gtfs/data
# - If GTFS_DATA_DIR env var is set, use that.
# - Otherwise, go up from current directory to project root, then to .src/gtfs/data
_current = Path.cwd().resolve()
_project_root = _current.parent if _current.name == "scripts" else _current

# Data folder location
DATA_DIR = Path(os.environ.get("GTFS_DATA_DIR", str(_project_root / ".src" / "gtfs" / "data"))).resolve()

# Set this to a filename like 'trips.txt' to convert only one file.
# Set it to None to convert every .txt file in DATA_DIR.
TXT_FILE_NAME = "stop_times_cleaned.txt"

# Excel limits one sheet to 1,048,576 rows total, including the header.
EXCEL_MAX_ROWS = 800_000
EXCEL_MAX_DATA_ROWS = EXCEL_MAX_ROWS - 1

# Output folder for generated Excel files.
OUTPUT_DIR = DATA_DIR / "excel_exports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def get_txt_files() -> list[Path]:
    all_txt_files = sorted(DATA_DIR.glob("*.txt"))
    if not all_txt_files:
        raise FileNotFoundError(
            f"No .txt files were found in {DATA_DIR}. "
            "Set GTFS_DATA_DIR if your data is in a different location."
        )

    if TXT_FILE_NAME is None:
        return all_txt_files

    selected_file = (DATA_DIR / TXT_FILE_NAME).resolve()
    if not selected_file.exists():
        raise FileNotFoundError(
            f"Selected file not found: {selected_file}. "
            "Update TXT_FILE_NAME to a valid .txt filename."
        )

    return [selected_file]


txt_files = get_txt_files()

print(f"Input folder: {DATA_DIR}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Conversion mode: {'all .txt files' if TXT_FILE_NAME is None else 'single file'}")
for file_path in txt_files:
    print(f" - {file_path.name}")

Input folder: C:\Users\Sarad\Escritorio\Mates\Cursos\Curs 2025-2026 (3r + 4t)\TFG\TFG\.src\gtfs\data
Output folder: C:\Users\Sarad\Escritorio\Mates\Cursos\Curs 2025-2026 (3r + 4t)\TFG\TFG\.src\gtfs\data\excel_exports
Conversion mode: single file
 - stop_times_cleaned.txt


In [2]:
def read_txt_table(file_path: Path) -> pd.DataFrame:
    """Read a GTFS TXT file using a robust fallback for encoding."""
    try:
        return pd.read_csv(file_path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(file_path, low_memory=False, encoding="latin1")

def write_excel_with_auto_split(df: pd.DataFrame, output_file: Path) -> list[Path]:
    """Write one Excel file or split the data into multiple files if needed."""
    if len(df) <= EXCEL_MAX_DATA_ROWS:
        df.to_excel(output_file, index=False)
        return [output_file]

    output_files: list[Path] = []
    total_parts = math.ceil(len(df) / EXCEL_MAX_DATA_ROWS)
    for part_index in range(total_parts):
        start_row = part_index * EXCEL_MAX_DATA_ROWS
        end_row = start_row + EXCEL_MAX_DATA_ROWS
        part_df = df.iloc[start_row:end_row]
        part_file = output_file.with_name(f"{output_file.stem}_part{part_index + 1}{output_file.suffix}")
        part_df.to_excel(part_file, index=False)
        output_files.append(part_file)

    return output_files


conversion_log = []

for txt_file in txt_files:
    df = read_txt_table(txt_file)
    output_file = OUTPUT_DIR / f"{txt_file.stem}.xlsx"
    output_files = write_excel_with_auto_split(df, output_file)

    conversion_log.append(
        (
            txt_file.name,
            ", ".join(path.name for path in output_files),
            len(df),
            len(df.columns),
        )
    )
    print(f"Converted {txt_file.name} -> {', '.join(path.name for path in output_files)}")

summary_df = pd.DataFrame(
    conversion_log,
    columns=["input_txt", "output_xlsx", "rows", "columns"],
)
summary_df

Converted stop_times_cleaned.txt -> stop_times_cleaned_part1.xlsx, stop_times_cleaned_part2.xlsx


,input_txt,output_xlsx,rows,columns
0,stop_times_cleaned.txt,"stop_times_cleaned_part1.xlsx, stop_times_clea...",1060371,5


In [ ]:
def excel_sheet_name(base_name: str, used_names: set[str]) -> str:
    """Create a valid, unique Excel sheet name (max 31 chars)."""
    cleaned = base_name.replace("/", "_").replace("\\", "_").replace("*", "_")
    cleaned = cleaned.replace("?", "_").replace("[", "_").replace("]", "_").replace(":", "_")
    cleaned = cleaned[:31] if cleaned else "Sheet"

    candidate = cleaned
    counter = 1
    while candidate in used_names:
        suffix = f"_{counter}"
        candidate = f"{cleaned[:31-len(suffix)]}{suffix}"
        counter += 1

    used_names.add(candidate)
    return candidate


def add_dataframe_to_writer(
    writer: pd.ExcelWriter,
    df: pd.DataFrame,
    base_name: str,
    used_sheet_names: set[str],
) -> list[str]:
    """Write a DataFrame to one or more sheets when the Excel row limit is exceeded."""
    if len(df) <= EXCEL_MAX_DATA_ROWS:
        sheet_name = excel_sheet_name(base_name, used_sheet_names)
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        return [sheet_name]

    total_parts = math.ceil(len(df) / EXCEL_MAX_DATA_ROWS)
    sheet_names: list[str] = []
    for part_index in range(total_parts):
        start_row = part_index * EXCEL_MAX_DATA_ROWS
        end_row = start_row + EXCEL_MAX_DATA_ROWS
        part_df = df.iloc[start_row:end_row]
        part_base_name = f"{base_name}_part{part_index + 1}"
        sheet_name = excel_sheet_name(part_base_name, used_sheet_names)
        part_df.to_excel(writer, sheet_name=sheet_name, index=False)
        sheet_names.append(sheet_name)

    return sheet_names


combined_file = OUTPUT_DIR / "gtfs_all_tables.xlsx"
used_sheet_names: set[str] = set()

with pd.ExcelWriter(combined_file, engine="openpyxl") as writer:
    for txt_file in txt_files:
        df = read_txt_table(txt_file)
        sheet_names = add_dataframe_to_writer(writer, df, txt_file.stem, used_sheet_names)
        print(f"Added {txt_file.name} as: {', '.join(sheet_names)}")

print(f"Combined workbook created: {combined_file}")